In [1]:
import calendar
import pandas as pd
from datetime import date, timedelta
from sqlalchemy import create_engine
from pandas.tseries.offsets import BDay

engine = create_engine("sqlite:///c:\\ruby\\port_lite\\db\\development.sqlite3")
conlite = engine.connect()

engine = create_engine("mysql+pymysql://root:@localhost:3306/stock")
const = engine.connect()

pd.set_option('display.max_row',None)

data_path = "../data/"
csv_path = "\\Users\\User\\iCloudDrive\\"
box_path = "\\Users\\User\\Dropbox\\"
one_path = "\\Users\\User\\OneDrive\\Documents\\Data\\"

today = date.today()
yesterday = today - timedelta(days=1)
today, yesterday

(datetime.date(2023, 1, 7), datetime.date(2023, 1, 6))

In [2]:
# specify the number of business days
num_days = 1
# convert the timedelta object to a BusinessDay object
num_business_days = BDay(num_days)
yesterday = today - num_business_days
#yesterday = yesterday.date()
print(f'today: {today}')
print(f'yesterday: {yesterday}')

today: 2023-01-07
yesterday: 2023-01-06 00:00:00


In [3]:
yesterday = yesterday.date()
a_year_ago = yesterday - timedelta(days=365)
yesterday, a_year_ago

(datetime.date(2023, 1, 6), datetime.date(2022, 1, 6))

### Restart and Run All Cells

### Weekly process or when stataus changes.

In [4]:
pd.read_sql_query('SELECT * FROM sales ORDER BY name', conlite)

,id,name,fm_date,to_date,days,fm_price,to_price,diff,pct,ttl_spread,avg_spread,max_price,min_price,qty,created_at,updated_at,latest_date_id
0,1200124,AP,2021-12-30,2022-01-06,5,9.55,9.45,-0.10,-1.05,-2,-1,9.55,9.35,65485452,2023-01-02 15:34:14.926824,2023-01-02 15:34:14.926824,1
1,1200125,AP,2022-01-06,2022-01-10,2,9.45,9.70,0.25,2.65,5,2,9.90,9.55,30686539,2023-01-02 15:34:14.936040,2023-01-02 15:34:14.936040,1
2,1200126,AP,2022-01-10,2022-01-11,1,9.70,9.40,-0.30,-3.09,-6,-6,9.80,9.40,20966845,2023-01-02 15:34:14.945053,2023-01-02 15:34:14.945053,1
3,1200127,AP,2022-01-11,2022-01-14,3,9.40,9.55,0.15,1.60,3,1,9.60,9.45,22928325,2023-01-02 15:34:14.953115,2023-01-02 15:34:14.953115,1
4,1200128,AP,2022-01-13,2022-01-24,7,9.55,9.25,-0.30,-3.14,-6,-1,9.65,9.25,54615550,2023-01-02 15:34:14.961704,2023-01-02 15:34:14.961704,1
5,1200129,AP,2022-01-24,2022-01-27,3,9.25,10.10,0.85,9.19,16,5,10.20,9.25,79008096,2023-01-02 15:34:14.972568,2023-01-02 15:34:14.972568,1
6,1200130,AP,2022-01-27,2022-01-28,1,10.10,10.00,-0.10,-0.99,-1,-1,10.40,9.95,43295025,2023-01-02 15:34:14.981329,2023-01-02 15:34:14.981329,1
7,1200131,AP,2022-01-28,2022-02-02,3,10.00,10.30,0.30,3.00,3,1,10.40,10.00,48054231,2023-01-02 15:34:14.990838,2023-01-02 15:34:14.990838,1
8,1200132,AP,2022-02-02,2022-02-03,1,10.30,10.10,-0.20,-1.94,-2,-2,10.50,10.10,21605578,2023-01-02 15:34:14.998673,2023-01-02 15:34:14.998673,1
9,1200133,AP,2022-02-03,2022-02-07,2,10.10,10.60,0.50,4.95,5,2,10.70,10.10,38102972,2023-01-02 15:34:15.007030,2023-01-02 15:34:15.007030,1


In [5]:
sqlDel = """DELETE FROM sales"""
rp = conlite.execute(sqlDel)
rp.rowcount

2197

In [6]:
sql = """
SELECT name
FROM orders 
ORDER BY name
"""
df = pd.read_sql(sql, conlite)

names = df['name'].values.tolist()
in_p = ", ".join(map(lambda name: "'%s'" % name, names))
in_p

"'ASP', 'BANPU', 'BCH', 'CKP', 'CPF', 'CPNREIT', 'GFPT', 'GVREIT', 'IVL', 'JMART', 'LPF', 'MAKRO', 'PTT', 'SCC', 'SINGER', 'SYNEX', 'TFFIF', 'TFG', 'TMT', 'WHAIR'"

In [7]:
len(in_p.split(','))

20

### Get past one year data

In [8]:
sql = """
SELECT * 
FROM price 
WHERE date >= '%s' AND name IN (%s) 
ORDER BY name, date"""
sql = sql % (a_year_ago, in_p)
print(sql)


SELECT * 
FROM price 
WHERE date >= '2022-01-06' AND name IN ('ASP', 'BANPU', 'BCH', 'CKP', 'CPF', 'CPNREIT', 'GFPT', 'GVREIT', 'IVL', 'JMART', 'LPF', 'MAKRO', 'PTT', 'SCC', 'SINGER', 'SYNEX', 'TFFIF', 'TFG', 'TMT', 'WHAIR') 
ORDER BY name, date


In [9]:
df = pd.read_sql(sql, const)
df.shape

(4860, 7)

In [10]:
data_path = "../data/"
file_name = "Yearly-Price-by-Name.csv"
output_file = data_path + file_name

df.set_index("name", inplace=True)
df.to_csv(output_file, header=None)

In [11]:
sql = """
SELECT name,trade 
FROM orders 
WHERE name IN (%s)
ORDER BY name
"""
sql = sql % in_p
orders = pd.read_sql(sql, conlite)
orders.set_index(['name'],inplace=True)
orders.shape

(20, 1)

### Create monitors from stocks

In [12]:
sql = """
SELECT name, trade 
FROM orders 
ORDER BY name
"""
monitors = pd.read_sql(sql, conlite)
monitors.set_index(["name"], inplace=True)
monitors

,trade
name,
ASP,S
BANPU,S
BCH,S
CKP,B
CPF,B
CPNREIT,B
GFPT,B
GVREIT,B
IVL,B


In [13]:
monitors.shape[0]

20

In [14]:
file_name = "monitors.csv"
data_file = data_path + file_name
output_file = csv_path + file_name
box_file = box_path + file_name

orders.to_csv(data_file, header=None)
orders.to_csv(output_file)
orders.to_csv(box_file)

In [15]:
sql = """
SELECT trade, COUNT(*) AS items 
FROM orders
GROUP BY trade
ORDER BY trade
"""
grp = pd.read_sql(sql, conlite)
grp

,trade,items
0,B,12
1,S,8


In [16]:
file_name = 'price-uploads.csv'
url = data_path + file_name
url

'../data/price-uploads.csv'

In [17]:
uploads = pd.read_csv(url)
uploads.sort_values(['name'],ascending=[True]).shape

(145, 1)

In [18]:
df_merge = pd.merge(orders, uploads, on='name', how='outer', indicator=True)
df_merge.sort_values(['name'],ascending=[True]).shape

(149, 3)

In [19]:
new_prices = df_merge['_merge'] == 'left_only'
df_merge[new_prices]

,name,trade,_merge
4,CPF,B,left_only
5,CPNREIT,B,left_only
16,TFFIF,B,left_only
17,TFG,B,left_only


In [20]:
new_prices = df_merge['_merge'] == 'right_only'
df_merge[new_prices].shape

(129, 3)

In [21]:
sql = """
SELECT name, status, market
FROM stocks 
ORDER BY status, name
"""
stocks = pd.read_sql(sql, conlite)
stocks.set_index(["name"], inplace=True)
stocks.shape

(67, 2)

In [22]:
file_name = "stocks-all.csv"
data_file = data_path + file_name
output_file = csv_path + file_name
box_file = box_path + file_name
one_file = one_path + file_name

stocks.to_csv(data_file, header=None)
stocks.to_csv(output_file)
stocks.to_csv(box_file)
stocks.to_csv(one_file)

In [23]:
sql = """
SELECT name, status 
FROM stocks 
WHERE status IN ("B","O")
ORDER BY name
"""

buy_candidates = pd.read_sql(sql, conlite)
buy_candidates.set_index(["name"], inplace=True)
buy_candidates

,status
name,
CKP,O
CPF,O
CPNREIT,B
GFPT,O
GVREIT,B
IVL,B
LPF,O
PTT,O
SINGER,B


In [24]:
buy_candidates.shape[0]

12

In [25]:
sql = """
SELECT name, status 
FROM stocks 
WHERE status IN ("I","S")
ORDER BY name
"""

sell_candidates = pd.read_sql(sql, conlite)
sell_candidates.set_index(["name"], inplace=True)
sell_candidates

,status
name,
ASP,S
BANPU,I
BCH,I
JMART,I
MAKRO,I
SCC,I
SYNEX,I
TMT,S


In [26]:
sell_candidates.shape[0]

8